# 02 · Explore player contributions

**Goal:** understand the StatsBomb sample and produce a descriptive player table before designing recruitment comparisons.

Run this after `01_setup.ipynb`. No additional packages are needed. Run cells in order with Shift+Enter. We will inspect the actual matches, aggregate contributions, and examine how playing time and small samples limit interpretation.

This notebook creates `reports/event_player_exploration.csv`. It does not train a model or rank player ability.


In [ ]:
import sys
import csv
import json
from collections import Counter, defaultdict
from pathlib import Path

assert sys.version_info >= (3, 11), "Use a Python 3.11+ kernel."
PROJECT_OVERRIDE = None  # Optional: Path("/path/to/soccer-recruitment")
if PROJECT_OVERRIDE is not None:
    candidates = [Path(PROJECT_OVERRIDE).expanduser()]
else:
    candidates = []
    for folder in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([folder, folder / "soccer-recruitment"])
ROOT = next((p.resolve() for p in candidates if (p / "src/soccer/cli.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory, or set PROJECT_OVERRIDE to your complete checkout.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
for folder in ("reports", "data/processed"):
    (ROOT / folder).mkdir(parents=True, exist_ok=True)
print("Python:", sys.version.split()[0])
print("Project:", ROOT.name)

paths = sorted((ROOT / "data/raw/statsbomb/events").glob("*.json"))
if not paths:
    raise FileNotFoundError("Run notebook 01 with acquisition enabled first.")
print("Downloaded event matches:", len(paths))


## 1. Identify the cohort

Before comparing players, confirm competition, season, match dates, and teams. This notebook analyzes **all downloaded event matches** in this project. The starter should have three matches; expanding acquisition changes the cohort. The downloaded match catalog can contain more matches than the downloaded event files.


In [ ]:
matches = {}
for path in sorted((ROOT / "data/raw/statsbomb/matches").glob("*.json")):
    for match in json.loads(path.read_text(encoding="utf-8")):
        matches[str(match["match_id"])] = match

events_by_match = {}
for path in paths:
    events_by_match[path.stem] = json.loads(path.read_text(encoding="utf-8"))
    match = matches.get(path.stem)
    if match is None:
        raise ValueError(f"No match metadata for {path.stem}")
    print(f"{path.stem} | {match['match_date']} | "
          f"{match['competition']['competition_name']} {match['season']['season_name']} | "
          f"{match['home_team']['home_team_name']} vs {match['away_team']['away_team_name']} | "
          f"{len(events_by_match[path.stem]):,} events")
print("Total events:", sum(map(len, events_by_match.values())))


## 2. Inspect event coverage

Frequent event types tell us what this dataset records. An event count is not playing time: a player may spend many minutes making useful movements without touching the ball. StatsBomb events and the anonymized Metrica sample remain separate evidence streams.


In [ ]:
event_types = Counter(e["type"]["name"] for events in events_by_match.values() for e in events)
for name, count in event_types.most_common(15):
    print(f"{name:22s} {count:6,d}")


## 3. Build a descriptive player table

Group by stable player and team identifiers rather than player names. A player who changes teams would have separate rows. Count match appearances only when a player has a recorded action; this is not a lineup-based appearances or minutes calculation.

- **Pass completion:** a pass without a recorded outcome is completed under the provider's event convention.
- **xG:** sum of the provider's expected-goals estimates, not a model fitted here.
- **Pressures:** count of pressure events, not pressure effectiveness.
- **Starting roles:** context from Starting XI events only. These do not capture all in-match position changes or substitute roles.


In [ ]:
players = {}
seen_matches = defaultdict(set)
starting_roles = defaultdict(Counter)
for match_id, events in events_by_match.items():
    for event in events:
        if event["type"]["name"] == "Starting XI":
            for slot in event.get("tactics", {}).get("lineup", []):
                starting_roles[(event["team"]["id"], slot["player"]["id"])][slot["position"]["name"]] += 1
        if "player" not in event:
            continue
        key = (event["team"]["id"], event["player"]["id"])
        row = players.setdefault(key, dict(team_id=key[0], team=event["team"]["name"],
            player_id=key[1], player=event["player"]["name"], passes=0, completed_passes=0,
            shots=0, xg=0.0, pressures=0))
        seen_matches[key].add(match_id)
        kind = event["type"]["name"]
        if kind == "Pass":
            row["passes"] += 1
            row["completed_passes"] += int("outcome" not in event["pass"])
        elif kind == "Shot":
            row["shots"] += 1
            row["xg"] += event["shot"].get("statsbomb_xg", 0.0)
        elif kind == "Pressure":
            row["pressures"] += 1

rows = []
for key, row in sorted(players.items()):
    row = dict(row)
    row["matches_with_recorded_actions"] = len(seen_matches[key])
    row["starting_roles"] = "; ".join(sorted(starting_roles[key])) or "No starting role recorded"
    row["pass_completion_pct"] = 100 * row["completed_passes"] / row["passes"] if row["passes"] else None
    rows.append(row)
print("Player-team rows:", len(rows))
print("Players with only one recorded match:", sum(r["matches_with_recorded_actions"] == 1 for r in rows))


## 4. Examine passing with uncertainty

Restrict the display to at least 30 attempted passes, an illustrative filter rather than proof of sufficient evidence. We show players alphabetically to focus on interpretation.

The approximate 95% Wilson intervals describe uncertainty under an independent, same-probability pass model. Real passes differ in difficulty and are dependent within players, matches and possessions, so these intervals do **not** establish uncertainty in player ability or account for tactical context. A later model needs contextual adjustment and clustered evaluation.


In [ ]:
import math

def wilson_interval(successes, attempts, z=1.96):
    if not attempts:
        return None, None
    p = successes / attempts
    denominator = 1 + z*z / attempts
    center = (p + z*z / (2*attempts)) / denominator
    half = z * math.sqrt(p*(1-p)/attempts + z*z/(4*attempts*attempts)) / denominator
    return 100*(center-half), 100*(center+half)

MIN_PASSES = 30
eligible = sorted((r for r in rows if r["passes"] >= MIN_PASSES), key=lambda r: (r["team"], r["player"]))
print(f"Players meeting {MIN_PASSES} attempts: {len(eligible)}; showing first 15 alphabetically.")
print(f"{'Player':30} {'Passes':>6} {'Complete':>9} {'Approx. 95% interval':>22}")
for row in eligible[:15]:
    low, high = wilson_interval(row["completed_passes"], row["passes"])
    print(f"{row['player'][:30]:30} {row['passes']:6} {row['pass_completion_pct']:8.1f}% {low:8.1f}%–{high:5.1f}%")


## 5. Check the aggregation and save the table

Reconcile player totals against the raw events so accidental omissions or double-counting surface early. This verifies arithmetic, not suitability for scouting. The output contains raw counts; avoid comparing those counts as if everyone had equal exposure.


In [ ]:
all_events = [e for events in events_by_match.values() for e in events]
for event_type, column in [("Pass", "passes"), ("Shot", "shots"), ("Pressure", "pressures")]:
    expected = sum(e["type"]["name"] == event_type and "player" in e for e in all_events)
    actual = sum(r[column] for r in rows)
    assert actual == expected, (column, actual, expected)
assert all(0 <= r["completed_passes"] <= r["passes"] for r in rows)
expected_xg = sum(e["shot"].get("statsbomb_xg", 0) for e in all_events if e["type"]["name"] == "Shot" and "player" in e)
assert math.isclose(sum(r["xg"] for r in rows), expected_xg)

output = ROOT / "reports/event_player_exploration.csv"
output.parent.mkdir(parents=True, exist_ok=True)
with output.open("w", newline="", encoding="utf-8") as stream:
    writer = csv.DictWriter(stream, fieldnames=list(rows[0]))
    writer.writeheader()
    writer.writerows(rows)
print("Aggregation checks passed.")
print("Saved:", output)


## Interpret before modeling

Consider these questions while reading the table:

1. Why might a center back complete more passes than an attacking midfielder without being a better passer?
2. What changes if one player has 20 minutes and another has 90?
3. Would a 90% completion rate on 10 passes give the same evidence as 90% on 200?
4. Which useful off-ball contributions are missing from an event-only table?

**Next milestone:** audit playing time using lineups, substitutions, dismissals and actual period durations; then build role-aware, exposure-adjusted profiles. Do not estimate minutes from a player's first and last event. Expand the cohort before making recruitment claims.

For a future prediction task, define the scouting date and target horizon before feature engineering. Features must precede that date, and train/test periods must be chronological. Never use future minutes or future-season performance in current scouting features. See [methodology](../docs/methodology.md).
